# Lecture 2 Exercises

## Neural Networks and Modern Optimization

This notebook follows the lecture 2 overview and turns the main ideas into a small regression case study. We will:

- connect linear regression to neural networks as learned feature maps
- build a small multilayer perceptron (MLP) in JAX
- train it with minibatch SGD and Adam
- study validation loss, early stopping, and weight decay

The setting is intentionally small enough to run on a laptop CPU while still showing the main ideas from the lecture.

Note: if you are having trouble installing JAX, see the `uv_environment_setup.md` file. Alternatively, you could use Google Colab (free of charge).


In [ ]:
import os
from pathlib import Path

cache_root = Path.cwd() / ".notebook_cache"
mpl_config_dir = cache_root / "matplotlib"
xdg_cache_dir = cache_root / "xdg"
mpl_config_dir.mkdir(parents=True, exist_ok=True)
xdg_cache_dir.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(mpl_config_dir)
os.environ["XDG_CACHE_HOME"] = str(xdg_cache_dir)

import numpy as np
import matplotlib.pyplot as plt
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

style_name = "seaborn-v0_8-whitegrid"
plt.style.use(style_name if style_name in plt.style.available else "default")
np.set_printoptions(precision=3, suppress=True)


def mse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return np.mean((y_true - y_pred) ** 2)


def linear_design_matrix(x):
    x = np.asarray(x).reshape(-1, 1)
    return np.hstack([np.ones((x.shape[0], 1)), x])


def fit_linear_closed_form(x, y):
    X = linear_design_matrix(x)
    y = np.asarray(y).reshape(-1)
    return np.linalg.solve(X.T @ X, X.T @ y)


def predict_linear(theta, x):
    return linear_design_matrix(x) @ np.asarray(theta).reshape(-1)


def make_nonlinear_data(n=64, noise_std=0.12, seed=21):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(-1.5, 1.5, size=n))
    y_true = np.sin(2.3 * x) + 0.25 * x ** 2 - 0.15 * x
    y = y_true + rng.normal(0.0, noise_std, size=n)
    return x[:, None], y, y_true


def train_valid_test_split(x, y, train_size=28, valid_size=18, seed=8):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(x))
    train_idx = indices[:train_size]
    valid_idx = indices[train_size : train_size + valid_size]
    test_idx = indices[train_size + valid_size :]
    return (
        x[train_idx],
        y[train_idx],
        x[valid_idx],
        y[valid_idx],
        x[test_idx],
        y[test_idx],
    )


def batch_iterator(x, y, batch_size=16, seed=0):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(x))
    for start in range(0, len(x), batch_size):
        batch_idx = indices[start : start + batch_size]
        yield x[batch_idx], y[batch_idx]


def parameter_norm(params):
    return float(sum(jnp.linalg.norm(layer["W"]) for layer in params["layers"]))


## A quick JAX orientation

Why use JAX here?

- Modern machine learning is mostly array computing + automatic differentiation + fast hardware.
- `jax.numpy` behaves a lot like NumPy, but JAX can also differentiate code with `jax.grad`, vectorize code with `jax.vmap`, and run the same code on CPUs, GPUs, or TPUs.
- Randomness is explicit: JAX uses random keys that you split and pass around, rather than a hidden global random state (as in numpy).
- `jax.vmap` means "vectorized map": you write a function for one example, and JAX automatically lifts it to work over a whole batch without an explicit Python loop.
- We use JAX in this notebook because it keeps the mathematics close to the code. The same ideas also transfer to PyTorch, TensorFlow, or other frameworks.

JAX typically operates on 'pytrees', which are arbitrary containers of array-like objects. Internally, JAX flattens and operates on the array objects in auto-differentiation or array operations. An pytree could be e.g.

```
params = {
    "layer1": {
        "w": jnp.array([[...]]),   # shape (d_in, d_hidden1)
        "b": jnp.array([...]),     # shape (d_hidden1,)
    },
    "layer2": {
        "w": jnp.array([[...]]),   # shape (d_hidden1, d_hidden2)
        "b": jnp.array([...]),
    },
    "output": {
        "w": jnp.array([[...]]),   # shape (d_hidden2, d_out)
        "b": jnp.array([...]),
    }
}
```

which in this case represents a shallow neural network (i.e. the weights and biases that we need to optimise given a dataset). All that remains is a function that uses these parameters in a forward-pass.

In [ ]:
# Some JAX basics
intro_key = jax.random.PRNGKey(0)
intro_key, subkey = jax.random.split(intro_key)
intro_sample = jax.random.normal(subkey, shape=(3,))

intro_x = jnp.array([-1.0, 0.5, 2.0])
intro_y = jnp.array([-0.7, 0.2, 1.8])


def scalar_model(weight, x_value):
    return weight * x_value


# Vectorize-map a function over some inputs, here copy the weight for each element of x input
batched_model = jax.vmap(scalar_model, in_axes=(None, 0))


def intro_loss(weight):
    preds = batched_model(weight, intro_x)
    return jnp.mean((preds - intro_y) ** 2)


intro_grad = jax.grad(intro_loss)

print("Random sample from an explicit key:", intro_sample)
print("Vectorized predictions at w = 1.0:", batched_model(1.0, intro_x))
print("Loss at w = 1.0:", float(intro_loss(1.0)))
print("Gradient at w = 1.0:", float(intro_grad(1.0)))
print("Example device:", jax.devices()[0])


In [ ]:
x_all, y_all, y_true_all = make_nonlinear_data()
x_train, y_train, x_valid, y_valid, x_test, y_test = train_valid_test_split(x_all, y_all)

grid = np.linspace(-1.5, 1.5, 400)[:, None]
y_true_grid = np.sin(2.3 * grid[:, 0]) + 0.25 * grid[:, 0] ** 2 - 0.15 * grid[:, 0]

theta_linear = fit_linear_closed_form(x_train, y_train)
y_linear_grid = predict_linear(theta_linear, grid)
linear_valid_mse = mse(y_valid, predict_linear(theta_linear, x_valid))
linear_test_mse = mse(y_test, predict_linear(theta_linear, x_test))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x_train[:, 0], y_train, color="tab:blue", label="train")
axes[0].scatter(x_valid[:, 0], y_valid, color="tab:orange", label="validation")
axes[0].scatter(x_test[:, 0], y_test, color="tab:green", label="test")
axes[0].plot(grid[:, 0], y_true_grid, color="black", linewidth=2, label="true signal")
axes[0].set_title("Synthetic nonlinear regression data")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

axes[1].scatter(x_train[:, 0], y_train, color="tab:blue", label="train")
axes[1].plot(grid[:, 0], y_true_grid, color="black", linewidth=2, label="true signal")
axes[1].plot(grid[:, 0], y_linear_grid, color="tab:red", linewidth=2, label="linear fit")
axes[1].set_title("Closed-form linear baseline")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Train/valid/test sizes: {len(x_train)}/{len(x_valid)}/{len(x_test)}")
print(f"Linear baseline validation MSE: {linear_valid_mse:.4f}")
print(f"Linear baseline test MSE: {linear_test_mse:.4f}")


## Exercise 1: From Linear Regression to Learned Basis Functions

Consider a one-hidden-layer neural network

$$f(x; 	\theta) = \sum_{j=1}^m v_j\,\phi(w_j x + b_j) + c.$$

Tasks:

1. Explain how this reduces to ordinary linear regression if there is no hidden layer and the activation is the identity.
2. Interpret the hidden-unit features $\phi(w_j x + b_j)$ as learned basis functions.
3. Explain why the optimization problem becomes non-convex once both the hidden-layer parameters $(w_j, b_j)$ and output weights $v_j$ are learned together.


### Your explanation

Write your answer here. Make the connection to feature maps explicit and should distinguish between linearity in the input and linearity in the parameters.


## Exercise 2: Build a JAX MLP

We will represent the network parameters as a pytree of the form

`{"layers": [{"W": ..., "b": ...}, ...]}`.

Use `jnp.tanh` as the hidden activation.

Tasks:

- complete `init_mlp`
- complete `mlp_forward`
- verify that the prediction vector has shape `(batch_size,)`


In [ ]:
toy_tree = {
    "left": jnp.ones((2, 3)),
    "right": [jnp.zeros(4), jnp.arange(2.0)],
}

print("Pytree leaf shapes:")
print(jax.tree.map(lambda leaf: leaf.shape, toy_tree))

probe_x = jnp.linspace(-1.5, 1.5, 6).reshape(-1, 1)
print("Probe batch shape:", probe_x.shape)


In [ ]:
def glorot_limit(in_dim, out_dim):
    return jnp.sqrt(6 / (in_dim + out_dim))


def init_mlp(layer_sizes, seed=0):
    keys = jax.random.split(jax.random.PRNGKey(seed), len(layer_sizes) - 1)
    layers = []

    for key, in_dim, out_dim in zip(keys, layer_sizes[:-1], layer_sizes[1:]):
        limit = glorot_limit(in_dim, out_dim)
        W = jax.random.uniform(key, (in_dim, out_dim), minval=-limit, maxval=limit)
        b = jnp.zeros((out_dim,))
        layers.append({"W": W, "b": b}) # Dictionary for a layer's parameters

    return {"layers": layers}


def mlp_forward(params, x, activation=jnp.tanh):
    h = jnp.asarray(x)
    if h.ndim == 1:
        h = h[:, None]

    # TODO: apply affine map + activation through all hidden layers
    # TODO: apply the final affine layer without an activation
    # TODO: return a flat vector of predictions
    raise NotImplementedError("Implement the forward pass.")


In [ ]:
layer_sizes = [1, 32, 32, 1]
params = init_mlp(layer_sizes, seed=0)
probe_y = mlp_forward(params, probe_x)

print("Parameter tree:")
print(jax.tree.map(lambda leaf: leaf.shape, params))
print("Prediction shape:", probe_y.shape)

random_curve = mlp_forward(params, grid)
plt.figure(figsize=(7, 4))
plt.plot(grid[:, 0], random_curve, color="tab:purple", linewidth=2)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Random MLP before training")
plt.show()


## Exercise 3: Losses, Backpropagation, and Minibatch SGD

We now train the network with mean squared error.

Tasks:

- complete `mean_weight_squared` so it returns the average squared magnitude of the weight matrices
- complete `mse_loss`
- complete `sgd_update` using `jax.tree.map`
- run the training cell and compare the learned curve with the linear baseline

`run_training` is provided for you. It uses `jax.value_and_grad` to compute the gradients needed for backpropagation.


In [ ]:
def mean_weight_squared(params):
    # TODO: collect the weight matrices and return their mean squared entry value
    # This is the mean of the squares of all the weight values
    raise NotImplementedError("Compute the average squared weight magnitude.")


def mse_loss(params, x, y, weight_decay=0.0):
    y = jnp.asarray(y).reshape(-1)
    preds = mlp_forward(params, x)

    # TODO: compute the loss term
    # TODO: if weight_decay > 0, add weight_decay * mean_weight_squared(params)
    raise NotImplementedError("Implement the MSE objective with optional weight decay.")


loss_and_grad = jax.value_and_grad(mse_loss)


def sgd_update(params, grads, state, lr):
    # TODO: update every leaf with gradient descent and return (new_params, state)
    raise NotImplementedError("Implement one SGD step.")


def run_training(
    init_params,
    step_fn,
    init_state,
    x_train,
    y_train,
    x_valid,
    y_valid,
    *,
    epochs=300,
    batch_size=16,
    lr=1e-2,
    weight_decay=0.0,
    patience=None,
    seed=0,
):
    params = init_params
    state = init_state
    history = {"train_loss": [], "valid_loss": []}
    best_params = params
    best_valid_loss = np.inf
    best_epoch = -1
    wait = 0

    for epoch in range(epochs):
        for xb, yb in batch_iterator(x_train, y_train, batch_size=batch_size, seed=seed + epoch):
            _, grads = loss_and_grad(params, xb, yb, weight_decay)
            params, state = step_fn(params, grads, state, lr)

        train_loss = float(mse_loss(params, x_train, y_train, weight_decay=0.0))
        valid_loss = float(mse_loss(params, x_valid, y_valid, weight_decay=0.0))
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)

        if valid_loss < best_valid_loss - 1e-6:
            best_valid_loss = valid_loss
            best_epoch = epoch
            best_params = params
            wait = 0
        else:
            wait += 1

        if patience is not None and wait >= patience:
            break

    summary = {
        "best_epoch": best_epoch,
        "best_valid_loss": best_valid_loss,
        "stop_epoch": len(history["train_loss"]) - 1,
    }
    return best_params, history, summary


In [ ]:
sgd_init_params = init_mlp([1, 32, 32, 1], seed=0)
sgd_params, sgd_history, sgd_summary = run_training(
    sgd_init_params,
    sgd_update,
    None,
    x_train,
    y_train,
    x_valid,
    y_valid,
    epochs=300,
    batch_size=16,
    lr=0.03,
    seed=0,
)

y_sgd_grid = np.asarray(mlp_forward(sgd_params, grid))
sgd_test_mse = float(mse_loss(sgd_params, x_test, y_test, weight_decay=0.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x_train[:, 0], y_train, color="tab:blue", label="train")
axes[0].scatter(x_valid[:, 0], y_valid, color="tab:orange", label="validation")
axes[0].plot(grid[:, 0], y_true_grid, color="black", linewidth=2, label="true signal")
axes[0].plot(grid[:, 0], y_linear_grid, color="tab:red", linestyle="--", linewidth=2, label="linear baseline")
axes[0].plot(grid[:, 0], y_sgd_grid, color="tab:purple", linewidth=2, label="MLP with SGD")
axes[0].set_title("Linear model versus trained MLP")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

axes[1].plot(sgd_history["train_loss"], label="train loss")
axes[1].plot(sgd_history["valid_loss"], label="validation loss")
axes[1].set_title("SGD learning curves")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

print("SGD summary:", sgd_summary)
print(f"Linear baseline validation MSE: {linear_valid_mse:.4f}")
print(f"Best SGD validation MSE: {sgd_summary['best_valid_loss']:.4f}")
print(f"Best SGD test MSE: {sgd_test_mse:.4f}")


### Reflection questions

- Where does backpropagation enter this code, even though we never wrote out the derivatives by hand?
- Why is there no closed-form analogue of the normal equation for this MLP?
- Why do we use validation loss, rather than training loss, to choose the best epoch?


## Exercise 4: Adam and Optimizer Comparison

Adam combines momentum-like moving averages with coordinate-wise adaptive learning rates.

Short intuition:

- In plain SGD, every parameter is updated directly from the current gradient, using the same learning rate everywhere.
- In Adam, we also keep a running average of past gradients (`m`) and of squared gradients (`v`).
- The running average `m` acts a bit like momentum: it smooths noisy minibatch gradients and helps updates keep moving in useful directions.
- The running average `v` rescales each parameter's step size separately: directions with consistently large gradients get smaller effective steps, while directions with smaller gradients can keep larger ones.

So a good interpretation is that Adam is like SGD with two extras: memory of past gradients and an adaptive per-parameter learning rate.

The standard Adam update equations are:

$$g_t = \nabla_\theta \mathcal{L}(\theta_t)$$

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$

$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_{t+1} = \theta_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t} + \varepsilon}$$

Here `g_t` is the gradient at step `t`, `m_t` and `v_t` are the first and second moment estimates, `eta` is the learning rate, and `eps` is a small constant for numerical stability. You can compare this directly to the SGD gradient update in the lecture.

Tasks:

- understand the JAX code for `init_adam_state`
- understand the JAX code for `adam_update`
- train the same network with Adam
- compare Adam and SGD in terms of convergence speed and best validation performance


In [ ]:
def init_adam_state(params):
    # Hint: Adam keeps extra running statistics for every parameter tensor.
    # `m` is the first-moment estimate (running average of gradients),
    # `v` is the second-moment estimate (running average of squared gradients),
    # and `t` is the update counter, which usually starts at 0 and becomes 1
    # on the first optimizer step for the bias-correction formulas.
    zeros = jax.tree.map(jnp.zeros_like, params)
    return {"m": zeros, "v": zeros, "t": 0}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    # Hint: increment `t` first, then use that new value in the bias correction:
    # m_hat = m / (1 - beta1**t), v_hat = v / (1 - beta2**t).
    t = state["t"] + 1
    m = jax.tree.map(lambda m_prev, g: beta1 * m_prev + (1 - beta1) * g, state["m"], grads)
    v = jax.tree.map(lambda v_prev, g: beta2 * v_prev + (1 - beta2) * (g ** 2), state["v"], grads)

    m_hat = jax.tree.map(lambda m_leaf: m_leaf / (1 - beta1 ** t), m)
    v_hat = jax.tree.map(lambda v_leaf: v_leaf / (1 - beta2 ** t), v)

    new_params = jax.tree.map(
        lambda p, m_leaf, v_leaf: p - lr * m_leaf / (jnp.sqrt(v_leaf) + eps),
        params,
        m_hat,
        v_hat,
    )
    new_state = {"m": m, "v": v, "t": t}
    return new_params, new_state

In [ ]:
adam_init_params = init_mlp([1, 32, 32, 1], seed=0)

adam_state = init_adam_state(adam_init_params)

adam_params, adam_history, adam_summary = run_training(
    adam_init_params,
    adam_update,
    adam_state,
    x_train,
    y_train,
    x_valid,
    y_valid,
    epochs=300,
    batch_size=16,
    lr=0.01,
    seed=0,
)

y_adam_grid = np.asarray(mlp_forward(adam_params, grid))
adam_test_mse = float(mse_loss(adam_params, x_test, y_test, weight_decay=0.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sgd_history["valid_loss"], label="SGD validation")
axes[0].plot(adam_history["valid_loss"], label="Adam validation")
axes[0].set_title("Validation loss by optimizer")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].scatter(x_train[:, 0], y_train, color="tab:blue", label="train")
axes[1].plot(grid[:, 0], y_true_grid, color="black", linewidth=2, label="true signal")
axes[1].plot(grid[:, 0], y_sgd_grid, color="tab:purple", linewidth=2, label="SGD")
axes[1].plot(grid[:, 0], y_adam_grid, color="tab:olive", linewidth=2, label="Adam")
axes[1].set_title("Best-fit curves by optimizer")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()

print(
    f"SGD:  best epoch = {sgd_summary['best_epoch']}, best valid MSE = {sgd_summary['best_valid_loss']:.4f}, test MSE = {sgd_test_mse:.4f}"
)
print(
    f"Adam: best epoch = {adam_summary['best_epoch']}, best valid MSE = {adam_summary['best_valid_loss']:.4f}, test MSE = {adam_test_mse:.4f}"
)


### Reflection questions

- Which optimizer reaches a low validation loss faster on this problem?
- Does the optimizer change the model class, or only the path we take through parameter space?
- Why can two optimizers with the same loss function land at different parameter values?


## Exercise 5: Early Stopping and Weight Decay

Regularization is about generalization, not just fitting the training set.

In this final exercise, use a slightly larger network and compare:

- an unregularized Adam run trained for the full number of epochs
- an Adam run with `weight_decay=0.01` and `patience=40`

Tasks:

1. Fill in the training loop below.
2. Record the best epoch, best validation loss, test loss, and weight norm for each run.
3. Compare the validation curves and the fitted functions.
4. Explain whether early stopping or weight decay seems to matter more on this split.


In [ ]:
regularization_configs = [
    {"label": "full Adam run", "weight_decay": 0.0, "patience": None},
    {"label": "weight decay + early stopping", "weight_decay": 0.01, "patience": 40},
]

regularization_results = []

for config in regularization_configs:
    params = init_mlp([1, 64, 64, 1], seed=0)
    state = init_adam_state(params)

    # TODO: train with Adam using the config values above
    # TODO: compute the best test MSE and the weight norm of the selected parameters
    # TODO: append a dictionary with label, params, history, best_epoch, best_valid_loss, test_mse, and weight_norm
    pass

assert regularization_results, "Fill in the loop above before continuing."
regularization_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for entry in regularization_results:
    axes[0].plot(entry["history"]["valid_loss"], label=entry["label"])

axes[0].set_title("Validation curves for regularization choices")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("validation MSE")
axes[0].legend()

axes[1].scatter(x_train[:, 0], y_train, color="tab:blue", label="train")
axes[1].plot(grid[:, 0], y_true_grid, color="black", linewidth=2, label="true signal")

for entry in regularization_results:
    y_grid = np.asarray(mlp_forward(entry["params"], grid))
    axes[1].plot(grid[:, 0], y_grid, linewidth=2, label=entry["label"])

axes[1].set_title("Selected models after regularization")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()

for entry in regularization_results:
    print(
        f"{entry['label']}: best epoch = {entry['best_epoch']}, best valid MSE = {entry['best_valid_loss']:.4f}, test MSE = {entry['test_mse']:.4f}, ||W|| = {entry['weight_norm']:.4f}"
    )


## Optional extension

- Replace `tanh` with `relu` and compare the fitted curves.
- Make the network deeper by trying `[1, 32, 32, 32, 1]`.
- Try a different random seed for the train/validation split. Do the optimizer and regularization conclusions stay the same?


## Take-away questions

1. In what sense does an MLP generalize linear regression?
2. Why is backpropagation essential for training modern neural networks?
3. Why do we need iterative optimizers such as SGD or Adam once hidden layers are introduced?
4. What is the role of validation loss in optimizer comparison and early stopping?
5. How do early stopping and weight decay try to improve generalization?
